# Build Races Dimension

In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
circuits_df = spark.read.table(f"{catalog_name}.{silver_schema}.circuits").filter((F.col('batch_id') == v_batch_id))
races_df = spark.read.table(f"{catalog_name}.{silver_schema}.races").filter((F.col('batch_id') == v_batch_id))

In [0]:
races_dim = (
        races_df.join(
            circuits_df,
            races_df.circuit_id == circuits_df.circuit_id,
            'inner'
        ).select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

In [0]:
write_to_gold (
    df = races_dim,
    target_table = target_table,
    table_key = 's.season == t.season AND s.round == t.round',
    columns_to_update = [
        'season', 
        'round',
        'race_name', 
        'date', 
        'circuit_name', 
        'locality', 
        'country', 
    ]
)

In [0]:
# (
#     races_dim.write
#         .mode('overwrite')
#         .format('delta')
#         .saveAsTable(target_table)
# )


In [0]:
%sql
SELECT * FROM formula1.gold.dim_races 